In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal
from dotenv import load_dotenv
from pydantic import BaseModel, Field

In [2]:
from dotenv import load_dotenv
import os
from langchain_google_genai import ChatGoogleGenerativeAI

# Load environment variables from .env
load_dotenv()

# Initialize the LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",                   # correct parameter
    google_api_key=os.environ["GOOGLE_API_KEY"],  # correct parameter
    temperature=0
)

# Send a message
response = llm.invoke("Hi")
print(response.content)

Hi there! How can I help you today?


In [6]:
pr="""
can you give code of conditional chain in langraph,Return ONLY valid Python code.
Do not include explanations.
Do not include markdown.
Do not include comments outside code.
"""

In [9]:
a=llm.invoke(pr).content

In [10]:
clean_code = a.strip("'")   # remove outer quotes


In [11]:
clean_code = clean_code.encode().decode("unicode_escape")


In [12]:
clean_code

'import operator\nfrom typing import Annotated, Literal, TypedDict\n\nfrom langchain_core.messages import BaseMessage\nfrom langchain_core.runnables import RunnableBranch\nfrom langgraph.graph import StateGraph, END\n\n\nclass AgentState(TypedDict):\n    input: str\n    output: str\n    next_step: Literal["step_a", "step_b", "end"]\n\n\ndef step_a_node(state: AgentState) -> AgentState:\n    print("Executing Step A")\n    return {"output": f"Processed by A: {state[\'input\']}", "next_step": "end"}\n\n\ndef step_b_node(state: AgentState) -> AgentState:\n    print("Executing Step B")\n    return {"output": f"Processed by B: {state[\'input\']}", "next_step": "end"}\n\n\ndef router_node(state: AgentState) -> Literal["step_a", "step_b", "end"]:\n    print(f"Routing based on input: {state[\'input\']}")\n    if "A" in state["input"]:\n        return "step_a"\n    elif "B" in state["input"]:\n        return "step_b"\n    else:\n        return "end"\n\n\nworkflow = StateGraph(AgentState)\n\nwork

### multi-agent example using LangGraph

In [1]:
from typing import TypedDict

class GraphState(TypedDict):
    input: str
    plan: str
    result: str

In [ ]:
def planner_agent(state: GraphState) -> GraphState:
    user_input = state["input"]

    if "weather" in user_input.lower():
        plan = "get_weather"
    else:
        plan = "general_response"

    return {"plan": plan}

In [3]:
def executor_agent(state: GraphState) -> GraphState:
    plan = state["plan"]

    if plan == "get_weather":
        result = "Weather is sunny ☀️"
    else:
        result = "I can help with many things!"

    return {"result": result}

In [4]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(GraphState)

workflow.add_node("planner", planner_agent)
workflow.add_node("executor", executor_agent)

workflow.set_entry_point("planner")

workflow.add_edge("planner", "executor")
workflow.add_edge("executor", END)

app = workflow.compile()

In [5]:
response = app.invoke({"input": "Tell me the weather"})
print(response)

{'input': 'Tell me the weather', 'plan': 'get_weather', 'result': 'Weather is sunny ☀️'}


In [6]:
#################################################################################

### jUST FOR FUN

In [7]:
from typing import TypedDict

class GraphState(TypedDict):
    input: str
    plan: str
    result: str

In [8]:
def planner_agent(GraphState):
    user_input = GraphState["input"]

    if "weather" in user_input.lower():
        plan = "get_weather"
    else:
        plan = "general_response"

    return {"plan": plan}

In [9]:
state = {
    "input": "What is the weather today?",
    "plan": "",
    "result": ""
}

print(planner_agent(state))

{'plan': 'get_weather'}


In [10]:
# CONDITION 2

In [11]:
def planner_agent(GraphState) -> GraphState:
    user_input = GraphState["input"]

    if "weather" in user_input.lower():
        plan = "get_weather"
    else:
        plan = "general_response"

    return {"plan": plan}

In [12]:
state = {
    "input": "Hello",
    "plan": "",
    "result": ""
}

print(planner_agent(state))

{'plan': 'general_response'}


In [13]:
# ################################################3

In [14]:
"""1. `TypedDict` is a **static typing tool**, not real OOP — it only describes dictionary structure.
2. Python does **not enforce it at runtime**; it’s mainly for IDE support and static checkers like `mypy`.
3. In frameworks like LangGraph, it improves readability and state structure but still isn’t strict validation.
4. Avoid using the class name as a parameter (e.g., `def f(GraphState)`) because it shadows the type and causes confusion.
5. For true OOP with runtime validation and safer production code, prefer **`dataclass` or `pydantic`** over `TypedDict`.
"""

'1. `TypedDict` is a **static typing tool**, not real OOP — it only describes dictionary structure.\n2. Python does **not enforce it at runtime**; it’s mainly for IDE support and static checkers like `mypy`.\n3. In frameworks like LangGraph, it improves readability and state structure but still isn’t strict validation.\n4. Avoid using the class name as a parameter (e.g., `def f(GraphState)`) because it shadows the type and causes confusion.\n5. For true OOP with runtime validation and safer production code, prefer **`dataclass` or `pydantic`** over `TypedDict`.\n'